In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

sys.path.insert(0, str(Path.cwd().parent))  # points to src/
from shared_modeling import load_or_create_master_split_ids, run_model_experiment

In [2]:
columns = ['V2AF13', 'V2AF15', 'PublicID'] # father's age and years of education
df_paternal = pd.read_csv('../../Data/V2A.csv', usecols=columns)
df_outcomes = pd.read_csv('../../Data/Modified/Outcome.csv', usecols=['PublicID', 'MH_outcome'])

# Create the master split once and persist it for reuse in other notebooks.
split_path = 'master_split_ids.csv'
train_ids, test_ids = load_or_create_master_split_ids(df_outcomes, split_path)
df_outcomes

,PublicID,MH_outcome
0,00004O,1
1,00007I,1
2,00008G,0
3,00015J,0
4,00016H,1
...,...,...
7736,17349I,1
7737,17350A,1
7738,17351V,0
7739,17352T,1


In [3]:
df = pd.merge(df_paternal, df_outcomes, on='PublicID', how='inner')
df.shape

(7608, 4)

In [4]:
df = df[df != 'R']
df = df[df != 'D']
df['V2AF13'] = pd.to_numeric(df['V2AF13'], errors='coerce')
df['V2AF15'] = pd.to_numeric(df['V2AF15'], errors='coerce')
df

,PublicID,V2AF13,V2AF15,MH_outcome
0,00004O,26.0,5.0,1
1,00007I,39.0,3.0,1
2,00008G,32.0,3.0,0
3,00015J,33.0,6.0,0
4,00016H,29.0,3.0,1
...,...,...,...,...
7603,17349I,25.0,5.0,1
7604,17350A,38.0,3.0,1
7605,17351V,32.0,6.0,0
7606,17352T,31.0,6.0,1


In [5]:
X = df.drop(['MH_outcome', 'PublicID'], axis=1)
y = df['MH_outcome']

train_df = df[df['PublicID'].isin(train_ids)].copy()
test_df = df[df['PublicID'].isin(test_ids)].copy()

X_train = train_df.drop(['MH_outcome', 'PublicID'], axis=1)
X_test = test_df.drop(['MH_outcome', 'PublicID'], axis=1)
y_train = train_df['MH_outcome']
y_test = test_df['MH_outcome']

y.value_counts()

MH_outcome
0    4517
1    3091
Name: count, dtype: int64

In [6]:
X

,V2AF13,V2AF15
0,26.0,5.0
1,39.0,3.0
2,32.0,3.0
3,33.0,6.0
4,29.0,3.0
...,...,...
7603,25.0,5.0
7604,38.0,3.0
7605,32.0,6.0
7606,31.0,6.0


In [7]:
y.value_counts()

MH_outcome
0    4517
1    3091
Name: count, dtype: int64

In [8]:
# Run the LR pipeline with macro F1 grid search.
best_model, y_pred, metrics = run_model_experiment(
    X_train,
    X_test,
    y_train,
    y_test,
    'lr',
    ['V2AF13', 'V2AF15']
)

Dropping rows with missing values because impute=False (train: 196, test: 59).
Final dataset sizes for LR (impute=False): train=5888, test=1465
Fitting 5 folds for each of 30 candidates, totalling 150 fits
Best parameters found: {'classifier__C': 0.001, 'classifier__l1_ratio': 0.0}
Best CV Score (f1): 0.5057
Model Coefficients:
num__V2AF13: -0.031814528525676865
num__V2AF15: -0.17814671245571
Evaluation Metrics for LR with shared preprocessing and adaptive CV scoring:
Accuracy: 0.5802
Precision (positive class): 0.4792
Recall (positive class): 0.5486
F1 (positive class): 0.5115
Macro Precision: 0.5725
Macro Recall: 0.5750
Macro F1: 0.5717
ROC AUC: 0.5790


In [9]:
# Run the RF pipeline with macro F1 grid search.
best_model, y_pred, metrics = run_model_experiment(
    X_train,
    X_test,
    y_train,
    y_test,
    'rf',
    ['V2AF13', 'V2AF15']
)

Dropping rows with missing values because impute=False (train: 196, test: 59).
Final dataset sizes for RF (impute=False): train=5888, test=1465
Fitting 5 folds for each of 81 candidates, totalling 405 fits
Best parameters found: {'classifier__max_depth': 15, 'classifier__min_samples_leaf': 4, 'classifier__min_samples_split': 3, 'classifier__n_estimators': 700}
Best CV Score (f1): 0.4648
Feature Importances:
num__V2AF13: 0.6415585028201027
num__V2AF15: 0.35844149717989726
Evaluation Metrics for RF with shared preprocessing and adaptive CV scoring:
Accuracy: 0.5836
Precision (positive class): 0.4818
Recall (positive class): 0.5196
F1 (positive class): 0.5000
Macro Precision: 0.5714
Macro Recall: 0.5730
Macro F1: 0.5716
ROC AUC: 0.5824


In [10]:
# Run the XGBoost pipeline with macro F1 grid search.
best_model, y_pred, metrics = run_model_experiment(
    X_train,
    X_test,
    y_train,
    y_test,
    'xgb',
    ['V2AF13', 'V2AF15']
)

Dropping rows with missing values because impute=False (train: 196, test: 59).
Final dataset sizes for XGB (impute=False): train=5888, test=1465
Fitting 5 folds for each of 243 candidates, totalling 1215 fits
Best parameters found: {'classifier__colsample_bytree': 0.8, 'classifier__learning_rate': 0.05, 'classifier__max_depth': 4, 'classifier__n_estimators': 100, 'classifier__subsample': 0.7}
Best CV Score (f1): 0.4823
Feature Importances:
num__V2AF13: 0.35438448190689087
num__V2AF15: 0.6456155180931091
Evaluation Metrics for XGB with shared preprocessing and adaptive CV scoring:
Accuracy: 0.5761
Precision (positive class): 0.4728
Recall (positive class): 0.5026
F1 (positive class): 0.4872
Macro Precision: 0.5628
Macro Recall: 0.5639
Macro F1: 0.5630
ROC AUC: 0.5783


In [11]:
# Run the SVM pipeline with macro F1 grid search.
best_model, y_pred, metrics = run_model_experiment(
    X_train,
    X_test,
    y_train,
    y_test,
    'svm',
    ['V2AF13', 'V2AF15']
)

Dropping rows with missing values because impute=False (train: 196, test: 59).
Final dataset sizes for SVM (impute=False): train=5888, test=1465
Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best parameters found: {'classifier__estimator__C': 0.1, 'classifier__estimator__gamma': 0.1, 'classifier__estimator__kernel': 'rbf'}
Best CV Score (f1): 0.5033
Skipping feature-level SVM output to keep notebook output compact.
Evaluation Metrics for SVM with shared preprocessing and adaptive CV scoring:
Accuracy: 0.5877
Precision (positive class): 0.4868
Recall (positive class): 0.5349
F1 (positive class): 0.5097
Macro Precision: 0.5769
Macro Recall: 0.5790
Macro F1: 0.5770
ROC AUC: 0.5752
